<a href="https://colab.research.google.com/github/G0rav/machine_learning_explained_visually_free/blob/main/01-linear-regression/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression — run the video

Video Lesson - **[Watch on YouTube](https://youtu.be/aRXqBKdyTWA)**

Everything in the video, in the order it happened, so you can watch the numbers
come out. Then two more layers: things to change, and two problems with no cell
to run.

**You need:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`. Nothing else, and
nothing downloaded.

Run it top to bottom. The last cell prints a pass count — every section checks
itself against the figure that was on screen, so if something has drifted you
find out here rather than wondering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

CHECKS = []


def check(label, got, want, tol=1e-9):
    "Assert a value the video put on screen, and record the result."
    ok = abs(float(got) - float(want)) <= tol
    CHECKS.append((label, ok, got, want))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: got {got}, expected {want}")
    return ok

## Layer 1 — Follow along

### 1. The six cars

Six cars that have already sold. Ages in years, prices in thousands of dollars —
that's the unit on the chart's vertical axis all the way through the video, so
`23.2` is a twenty-three thousand two hundred dollar car.

These six are *placed*, not sampled: the ages and prices were chosen so the
arithmetic closes exactly. Mean age 5.0, mean price 14.0, and a best line of
`price = 24 - 2 x age` with no rounding anywhere.

In [ ]:
age   = np.array([1.0, 2.0, 4.0, 6.0, 8.0, 9.0])
price = np.array([23.2, 19.6, 15.2, 11.2, 7.6, 7.2])

QUESTION_AGE = 7.0          # the car the video opens on: seven years old, no price

cars = pd.DataFrame({"age_years": age, "price_k": price})
print(cars.to_string(index=False))
print()
check("mean age", age.mean(), 5.0)
check("mean price", price.mean(), 14.0)

The two lines the two people drew by eye. Neither is invented — each is exactly
the line through two of the six cars. Person A used the one- and six-year-old,
person B the four- and nine-year-old.

In [ ]:
def line_through(x, y, i, j):
    "The line through cars i and j, as (slope, intercept)."
    slope = (y[j] - y[i]) / (x[j] - x[i])
    return float(slope), float(y[i] - slope * x[i])


def predict(w, b, x):
    return np.asarray(x, float) * w + b


EYEBALL_A = line_through(age, price, 0, 3)     # the 1- and 6-year-old
EYEBALL_B = line_through(age, price, 2, 5)     # the 4- and 9-year-old

print(f"person A: slope {EYEBALL_A[0]:+.1f}, intercept {EYEBALL_A[1]:.1f}")
print(f"person B: slope {EYEBALL_B[0]:+.1f}, intercept {EYEBALL_B[1]:.1f}")
print()
check("A's answer at age 7 ($k)", predict(*EYEBALL_A, QUESTION_AGE), 8.8)
check("B's answer at age 7 ($k)", predict(*EYEBALL_B, QUESTION_AGE), 10.4)
check("they disagree by ($k)",
      predict(*EYEBALL_B, QUESTION_AGE) - predict(*EYEBALL_A, QUESTION_AGE), 1.6)

In [ ]:
# The cold open, as one picture: two reasonable lines, two different answers.
grid = np.linspace(0, 10, 100)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(age, price, s=70, zorder=3, label="cars that sold")
ax.plot(grid, predict(*EYEBALL_A, grid), label="person A")
ax.plot(grid, predict(*EYEBALL_B, grid), label="person B")
for (w, b), name in [(EYEBALL_A, "A"), (EYEBALL_B, "B")]:
    yq = predict(w, b, QUESTION_AGE)
    ax.plot([QUESTION_AGE], [yq], marker="o", ms=9, mfc="none", color="k", zorder=4)
    ax.annotate(f"{name}: ${yq * 1000:,.0f}", (QUESTION_AGE, yq),
                textcoords="offset points", xytext=(10, -4))
ax.axvline(QUESTION_AGE, ls=":", lw=1, color="grey")
ax.set(xlim=(0, 10), ylim=(0, 26),
       xlabel="age (years)", ylabel="price ($1,000s)")
ax.legend(loc="upper right")
ax.set_title("Same six cars, two honest lines, two different prices")
plt.show()

### 2. The constant baseline

The simplest model there is: ignore the age completely and quote the average
price for every car. It's the bar anything cleverer has to clear.

In [ ]:
baseline = price.mean()
print(f"the constant baseline predicts ${baseline * 1000:,.0f} for every car")


def residuals(w, b, x, y):
    "Actual minus predicted -- the *vertical* miss, which is the whole point."
    return np.asarray(y, float) - predict(w, b, x)


base_res = residuals(0.0, baseline, age, price)
print("its residuals:", base_res)
print()
check("baseline total squared error", float(base_res @ base_res), 212.48, tol=1e-9)
check("baseline total absolute miss", float(np.abs(base_res).sum()), 32.0)

### 2b. Does the model need `b`?

Force the line through the origin and let it pick the best slope it can from
there. Both numbers below are the same measure the video uses at this point:
every gap added up, signs dropped.


In [ ]:
# OLS with the intercept fixed at zero: w = sum(x*y) / sum(x*x)
w_origin = float((age * price).sum() / (age ** 2).sum())
print(f"best slope through the origin: w = {w_origin:+.4f}   (b forced to 0)")
print()

miss_origin   = float(np.abs(residuals(w_origin, 0.0, age, price)).sum())
miss_baseline = float(np.abs(base_res).sum())
print(f"total miss, six cars, signs ignored")
print(f"   no intercept  {miss_origin:6.1f}")
print(f"   the baseline  {miss_baseline:6.1f}")
print()
print("Take b away and the line does WORSE than ignoring age altogether -- and")
print("its slope comes out positive, claiming older cars are worth more.")
print("That is what b is for: it lets the line sit at the right height.")

check("best slope through the origin", w_origin, 1.5644, tol=5e-5)
check("no-intercept total miss", miss_origin, 60.7, tol=0.05)
check("baseline total miss", miss_baseline, 32.0)


### 3. Scoring a line

Four lines of code, and they are the definition the video boxed up: run the model
on every car, take the gap, square each one, add them up.

Note that `x` and `y` are NumPy arrays. That is what lets `w * x + b` work on all
six cars at once instead of looping, and it is why `.sum()` is available on the
result.

In [ ]:
def sse(w, b, x, y):
    predicted = w * x + b
    residual  = y - predicted
    return (residual ** 2).sum()


TILT_DEMO = (-2.1, 24.6)        # the tilted line from early in the video

print(f"person A       {sse(*EYEBALL_A, age, price):8.2f}")
print(f"person B       {sse(*EYEBALL_B, age, price):8.2f}")
print(f"the tilt line  {sse(*TILT_DEMO, age, price):8.2f}   <- the rule ranks it best of the three")
print(f"the baseline   {sse(0.0, baseline, age, price):8.2f}")
print()
check("person A's score", sse(*EYEBALL_A, age, price), 13.76, tol=1e-9)
check("person B's score", sse(*EYEBALL_B, age, price), 13.76, tol=1e-9)
check("the tilt line's score", sse(*TILT_DEMO, age, price), 5.06, tol=1e-9)

A and B score *exactly* the same, which is the thing the video notices and leaves
alone. It isn't a coincidence and it isn't a bug — hold on to it, because the
maths video explains it in one picture. Layer 3 asks you to find another pair
that ties.

### 4. Training, from scratch

Gradient descent. Nudge each parameter a hair and measure what the score does —
that pair of numbers is the **gradient**, and it points the way the cost rises
fastest. Subtract a small fraction of it and the cost falls. Repeat. The
fraction is the **learning
rate**, and it is the one number you have to choose yourself.

This is the whole of training, and it is the real method — not a stand-in.

In [ ]:
def gradient(w, b, x, y, h=1e-5):
    # Nudge each parameter and measure. This is exactly what the video shows.
    dw = (sse(w + h, b, x, y) - sse(w - h, b, x, y)) / (2 * h)
    db = (sse(w, b + h, x, y) - sse(w, b - h, x, y)) / (2 * h)
    return dw, db

LEARNING_RATE = 0.002
w_hat, b_hat = -0.5, 18.0                     # where the video starts iterating
trail = [(w_hat, b_hat, sse(w_hat, b_hat, age, price))]
for _ in range(20_000):
    dw, db = gradient(w_hat, b_hat, age, price)
    w_hat -= LEARNING_RATE * dw
    b_hat -= LEARNING_RATE * db
    trail.append((w_hat, b_hat, sse(w_hat, b_hat, age, price)))

print(f"the gradient where it started: {gradient(-0.5, 18.0, age, price)}")
print()
for step in (0, 1, 5, 25, 100, 500, 2000):
    w_s, b_s, c_s = trail[step]
    print(f"  step {step:5d}:  w = {w_s:8.4f}   b = {b_s:8.4f}   SSE = {c_s:9.2f}")
print()
print(f"from scratch    w = {w_hat:.4f}   b = {b_hat:.4f}   SSE = {sse(w_hat, b_hat, age, price):.2f}")
print()
fit_res = residuals(w_hat, b_hat, age, price)
print("its residuals:", np.round(fit_res, 4))
print()
check("the gradient at the start (w)", gradient(-0.5, 18.0, age, price)[0], 246.0, tol=1e-6)
check("the gradient at the start (b)", gradient(-0.5, 18.0, age, price)[1], 18.0, tol=1e-6)
check("gradient descent slope", w_hat, -2.0, tol=1e-6)
check("gradient descent intercept", b_hat, 24.0, tol=1e-6)
check("its residuals sum to zero", fit_res.sum(), 0.0, tol=1e-6)
check("its score", sse(w_hat, b_hat, age, price), 4.48, tol=1e-6)
check("the seven-year-old ($k)", predict(w_hat, b_hat, QUESTION_AGE), 10.0, tol=1e-6)

It misses **every** car — not one of them sits on the line — but it doesn't miss
any of them by much. That is the trade the objective was asking for.

### 5. Training, with scikit-learn

The thing that trips everybody up first time: features go in as a *two-dimensional*
array, one row per car and one column per feature. Six cars, one feature, so
`(6, 1)`.

In [ ]:
X = age.reshape(-1, 1)
print("X.shape:", X.shape)

model = LinearRegression().fit(X, price)
print(f"coef_     {model.coef_}       type: {type(model.coef_).__name__}")
print(f"intercept_ {model.intercept_:.4f}")
print()
print("From scratch    w = {:.4f}   b = {:.4f}".format(w_hat, b_hat))
print("scikit-learn    w = {:.4f}   b = {:.4f}".format(model.coef_[0], model.intercept_))
print()
check("six-car feature shape is (6, 1)", X.shape[1], 1)
check("rows in X", X.shape[0], 6)
check("sklearn agrees on the slope", model.coef_[0], w_hat, tol=1e-6)
check("sklearn agrees on the intercept", model.intercept_, b_hat, tol=1e-6)
# tol is 1e-6, not 1e-12: gradient descent *approaches* the minimum, so the two
# methods agree to about ten decimal places rather than to the bit. That gap is
# the honest difference between converging on an answer and solving for one.

CHECKS.append(("coef_ is a NumPy array", isinstance(model.coef_, np.ndarray),
               type(model.coef_).__name__, "ndarray"))
print(f"{'PASS' if isinstance(model.coef_, np.ndarray) else 'FAIL'}  "
      f"coef_ is a NumPy array: {type(model.coef_).__name__}")

Same numbers. Not close — the same. The library isn't doing anything our ten
lines didn't do; it's minimising the same objective with a much better solver.

`coef_` comes back wrapped in brackets because it's a NumPy array with **one
entry per feature**. One feature here, so one entry. Add a second and there'd be
two.

### 5b. A coefficient is nothing without its unit

Same six cars, age in **months** instead of years. Nothing about the world has
changed, so the line must not move either.


In [ ]:
age_months   = age * 12
model_months = LinearRegression().fit(age_months.reshape(-1, 1), price)
w_m, b_m = float(model_months.coef_[0]), float(model_months.intercept_)

print(f"years   w = {w_hat:+.4f} per year     b = {b_hat:.4f}")
print(f"months  w = {w_m:+.4f} per month    b = {b_m:.4f}")
print()
print(f"  {w_m:+.4f} thousand per month  =  ${abs(w_m) * 1000:,.2f} per month")
print(f"  x 12 months                  =  ${abs(w_m) * 12000:,.0f} per year   <- the same drop")
print()
identical = np.allclose(model_months.predict(age_months.reshape(-1, 1)),
                        model.predict(X))
print(f"every prediction identical: {identical}")
print()
print("-2 alone is not an answer. -$2,000 per year of age is.")

check("slope in months", w_m, -1 / 6, tol=1e-9)
check("months slope x 12 == years slope", w_m * 12, w_hat, tol=1e-6)
check("the unit does not move the intercept", b_m, b_hat, tol=1e-6)
CHECKS.append(("changing the unit changes no prediction", bool(identical),
               identical, True))


### 6. The lot — sixty cars

Six cars are enough to follow by hand and far too few to say how well a model
performs. So: sixty *simulated* cars, from a curved depreciation rule plus random
price variation. Not sixty real sales, and not a claim about any real market —
just a larger, messier sample to practise the measurements on.

The generator is reproduced here exactly, seed and all, so your numbers match the
video's to the last decimal.

In [ ]:
LOT_SEED, LOT_N = 20260809, 60

rng = np.random.default_rng(LOT_SEED)
_a = np.sort(rng.uniform(0.4, 14.0, LOT_N))
_expected_miles_k = 11.5 * _a
# The mileage draw is unused below, but it MUST happen: it advances the generator,
# and skipping it changes every price that follows.
_mileage_k = np.maximum(_expected_miles_k + rng.normal(0.0, 14.0, LOT_N), 0.8)
_excess = _mileage_k - _expected_miles_k
_true = 25.0 * np.exp(-0.105 * _a) - 0.085 * _excess
_p = _true + rng.normal(0.0, 0.09 * np.abs(_true) + 0.30, LOT_N)

lot_age   = np.round(_a, 1)
lot_price = np.round(np.maximum(_p, 0.8), 1)

print(f"{LOT_N} cars, ages {lot_age.min()}-{lot_age.max()} years, "
      f"prices ${lot_price.min()}k-${lot_price.max()}k")
print()
check("lot size", len(lot_age), 60)
check("youngest car", lot_age.min(), 0.8)
check("oldest car", lot_age.max(), 13.9)
check("cheapest car", lot_price.min(), 3.3)
check("dearest car", lot_price.max(), 23.2)

In [ ]:
X_lot = lot_age.reshape(-1, 1)
print("X_lot.shape:", X_lot.shape, "   (compare (6, 1) above -- different dataset)")

lot_model = LinearRegression().fit(X_lot, lot_price)
lot_w, lot_b = float(lot_model.coef_[0]), float(lot_model.intercept_)
print(f"coefficient {lot_w:.4f} thousand dollars per year  (${lot_w * 1000:,.0f}/year)")
print(f"intercept   {lot_b:.4f} thousand dollars           (${lot_b * 1000:,.0f})")
print()
check("lot feature shape is (60, 1)", X_lot.shape[0], 60)
check("lot slope", lot_w, -1.2285, tol=5e-5)
check("lot intercept", lot_b, 21.4246, tol=5e-5)

In [ ]:
# RMSE: square them all, average, then square-root to get back into dollars.
lot_res = residuals(lot_w, lot_b, lot_age, lot_price)
rmse = float(np.sqrt((lot_res ** 2).mean()))
inside = int((np.abs(lot_res) <= rmse).sum())
print(f"RMSE ${rmse * 1000:,.0f}   -- {inside} of {LOT_N} cars fall within one RMSE of the line")

# R-squared: the model's squared error against the constant baseline's.
lot_baseline = lot_price.mean()
ss_res = float(lot_res @ lot_res)
ss_tot = float(((lot_price - lot_baseline) ** 2).sum())
r2 = 1.0 - ss_res / ss_tot
print(f"baseline ${lot_baseline * 1000:,.0f} for every car")
print(f"baseline squared error {ss_tot:.2f}   model squared error {ss_res:.2f}")
print(f"ratio {ss_res / ss_tot:.4f}   ->   R-squared {r2:.4f}")
print()
check("RMSE ($k)", rmse, 1.799, tol=5e-4)
check("cars inside one RMSE", inside, 39)
check("lot mean price ($k)", lot_baseline, 11.783, tol=5e-4)
check("baseline squared error", ss_tot, 1596.38, tol=5e-3)
check("model squared error", ss_res, 194.18, tol=5e-3)
check("R-squared", r2, 0.8784, tol=5e-5)

In [ ]:
band = np.linspace(0, 15, 100)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(lot_age, lot_price, s=28, alpha=0.8, label="60 simulated cars")
ax.plot(band, predict(lot_w, lot_b, band), lw=2, label="fitted line")
ax.fill_between(band,
                predict(lot_w, lot_b, band) - rmse,
                predict(lot_w, lot_b, band) + rmse,
                alpha=0.15, label=f"+/- 1 RMSE ({inside}/60 inside)")
ax.axhline(lot_baseline, ls="--", lw=1.5, color="grey",
           label=f"constant baseline (${lot_baseline * 1000:,.0f})")
ax.set(xlim=(0, 15), ylim=(0, 26),
       xlabel="age (years)", ylabel="price ($1,000s)")
ax.legend(loc="upper right", fontsize=9)
ax.set_title(f"R-squared {r2:.4f}, measured on the cars it was fitted to")
plt.show()

Every number in that section was measured on the same sixty cars the model was
fitted to. So it says how well the line describes *this* data — not how it would
do on cars it has never met. Those are different questions.

### 7. Where it breaks

Two failures, both of them properties of the model's own two numbers.

In [ ]:
# (1) Extrapolation. The line has no idea what a car is, or that money stops at zero.
for a in [7.0, 14.0, 20.0, 25.0]:
    inside_data = lot_age.min() <= a <= lot_age.max()
    tag = "interpolation" if inside_data else "EXTRAPOLATION"
    print(f"age {a:5.1f} -> ${predict(lot_w, lot_b, a) * 1000:>9,.0f}   {tag}")
print()
check("prediction at 25 years ($k)", predict(lot_w, lot_b, 25.0), -9.287, tol=5e-4)

In [ ]:
# (2) The linearity assumption: a straight line means the SAME drop every year.
drops = np.diff(price) / np.diff(age)
for i, d in enumerate(drops):
    print(f"{age[i]:.0f} -> {age[i+1]:.0f} years:  ${d * 1000:+,.0f} per year")
print()
print("Not one constant number: a big drop early, almost nothing late.")
for i, want in enumerate([-3.6, -2.2, -2.0, -1.8, -0.4]):
    check(f"drop {i+1}", drops[i], want, tol=1e-9)

In [ ]:
# The curve the lot was generated from, against the straight line fitted to it.
curve_age = np.linspace(0.4, 14.0, 200)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(lot_age, lot_price, s=24, alpha=0.6, label="60 simulated cars")
ax.plot(curve_age, predict(lot_w, lot_b, curve_age), lw=2, label="straight fit")
ax.plot(curve_age, 25.0 * np.exp(-0.105 * curve_age), lw=2, ls="--",
        label="the curve they came from")
ax.set(xlim=(0, 15), ylim=(0, 26),
       xlabel="age (years)", ylabel="price ($1,000s)")
ax.legend(loc="upper right")
ax.set_title("Close through the middle, apart at both ends")
plt.show()

## Layer 2 — Experiment

Change something, re-run, and watch what moves. Each one has a note on what to
look for. Nothing below is checked — that's the point.

In [ ]:
# Change one car's AGE, refit, and see how far w and b move.
age2 = age.copy()
age2[0] = 3.0                      # the one-year-old becomes three years old
m = LinearRegression().fit(age2.reshape(-1, 1), price)
print(f"was  w = {w_hat:+.4f}  b = {b_hat:.4f}")
print(f"now  w = {m.coef_[0]:+.4f}  b = {m.intercept_:.4f}")
# Watch for: moving a car near the edge of the age range tilts the line most.

In [ ]:
# Change that car's PRICE instead. Which moves the line more -- age or price?
price2 = price.copy()
price2[0] = 18.0                   # the newest car sold cheap
m = LinearRegression().fit(X, price2)
print(f"now  w = {m.coef_[0]:+.4f}  b = {m.intercept_:.4f}")

In [ ]:
# Insert a corrupt row: $4.5k typed as $45.0k, on an OLD car. Then refit.
# This is where the words *outlier* and *influential observation* belong.
bad_age   = np.append(lot_age, 13.4)
bad_price = np.append(lot_price, 45.0)
m = LinearRegression().fit(bad_age.reshape(-1, 1), bad_price)
r2_bad = m.score(bad_age.reshape(-1, 1), bad_price)
print(f"clean:  w = {lot_w:+.4f}  b = {lot_b:.4f}  R2 = {r2:.4f}")
print(f"corrupt: w = {m.coef_[0]:+.4f}  b = {m.intercept_:.4f}  R2 = {r2_bad:.4f}")

# Now move the SAME bad row to the middle of the age range and refit.
mid_age   = np.append(lot_age, 7.0)
m2 = LinearRegression().fit(mid_age.reshape(-1, 1), bad_price)
print(f"same error at age 7: w = {m2.coef_[0]:+.4f}  b = {m2.intercept_:.4f}")
# Watch for: one wrong row does far more damage far from the mean age.
# That is leverage, and it has a video of its own.

In [ ]:
# Delete the three oldest cars and refit. How much of the slope came from them?
keep = np.argsort(lot_age)[:-3]
m = LinearRegression().fit(lot_age[keep].reshape(-1, 1), lot_price[keep])
print(f"all 60:      w = {lot_w:+.4f}")
print(f"without top 3: w = {m.coef_[0]:+.4f}")

In [ ]:
# Change the learning rate. This is the one number gradient descent makes you pick.
for lr in [0.00002, 0.002, 0.0048, 0.005, 0.02]:
    w_t, b_t = -0.5, 18.0
    for _ in range(20_000):
        dw, db = gradient(w_t, b_t, age, price)
        w_t -= lr * dw
        b_t -= lr * db
        if not np.isfinite(w_t):
            break
    if np.isfinite(w_t) and abs(w_t) < 1e6:
        tag = "converged" if abs(w_t + 2.0) < 1e-3 else "not converged"
        print(f"lr = {lr:<8}: w = {w_t:+10.4f}  b = {b_t:9.4f}   {tag}")
    else:
        print(f"lr = {lr:<8}: DIVERGED  ->  w = {w_t:.3e}")
# Watch for: there is a hard edge. This problem's stability limit is
# 2 / (largest curvature) = 2 / 412.98 = 0.004843, and you can see it -- 0.0048
# converges, 0.005 does not. Below the edge, smaller is not safer, only slower:
# 0.00002 has not converged after twenty thousand steps -- it has only reached
# w = -1.73. Choosing this number is
# the whole subject of the next video.

In [ ]:
# Fit with ABSOLUTE error instead of squares. How far does the line move?
# An absolute-error optimum always passes through two of the points, so checking
# all fifteen pairs solves it exactly -- no solver needed.
from itertools import combinations

best_abs = None
for i, j in combinations(range(6), 2):
    w, b = line_through(age, price, i, j)
    s = float(np.abs(residuals(w, b, age, price)).sum())
    if best_abs is None or s < best_abs[0]:
        best_abs = (s, w, b)
print(f"least squares:   w = {w_hat:+.1f}  b = {b_hat:.1f}")
print(f"absolute error:  w = {best_abs[1]:+.1f}  b = {best_abs[2]:.1f}")
print(f"they disagree about a brand-new car by "
      f"${abs(best_abs[2] - b_hat) * 1000:,.0f}")
# Watch for: same slope, different intercept. A different objective, a different line.

In [ ]:
# Change the lot's seed. How much do the coefficients move on 'the same' market?
for seed in [20260809, 1, 2, 3, 4]:
    r = np.random.default_rng(seed)
    a = np.sort(r.uniform(0.4, 14.0, LOT_N))
    mk = np.maximum(11.5 * a + r.normal(0.0, 14.0, LOT_N), 0.8)
    t = 25.0 * np.exp(-0.105 * a) - 0.085 * (mk - 11.5 * a)
    pr = np.round(np.maximum(t + r.normal(0.0, 0.09 * np.abs(t) + 0.30, LOT_N), 0.8), 1)
    m = LinearRegression().fit(np.round(a, 1).reshape(-1, 1), pr)
    print(f"seed {seed:>9}:  w = {m.coef_[0]:+.4f}  b = {m.intercept_:7.4f}")
# Watch for: the coefficient is an estimate. A different sample gives a different number.

## Layer 3 — Challenge

No cells. Work these out, then check yourself however you like.

**1. Build a dataset where linear regression does badly.**
Make one, fit it, and explain the failure from the drops between neighbouring
points rather than from the R-squared. If your explanation is "R-squared was low",
you have described the symptom.

**2. Person A and person B scored exactly the same — 13.76 each. Find another
pair of two-car lines that ties.**
There are fifteen lines through pairs of our six cars. Some of them tie. Work out
what the tied lines have in common before you go looking, and the search is
short. What you find is the reason the maths video opens where it does.

In [ ]:
print(f"{sum(1 for _, ok, *_ in CHECKS if ok)} / {len(CHECKS)} checks passed")
failed = [c[0] for c in CHECKS if not c[1]]
if failed:
    print("FAILED:")
    for name in failed:
        print("  -", name)
    raise AssertionError(f"{len(failed)} check(s) failed")
print("Every figure matches the video.")